# Policy PDF/HTML Graph Retrieval with NL-to-Cypher

This notebook queries the policy graph created by pdf_policy_transformation.ipynb.

Flow: discover the live Apache AGE schema, generate a read-only policy-specific Cypher query, validate it, execute it with a short-lived PostgreSQL connection, retry recoverable failures, use deterministic evidence search when needed, and answer only from returned evidence.

Run pdf_policy_transformation.ipynb first whenever source PDFs/HTML change.

In [ ]:
# Install only if these packages are missing.
%pip install openai "psycopg[binary]" python-dotenv

In [1]:
import json
import os
import re
from collections import OrderedDict

import psycopg
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv(override=True)

PG_HOST = os.getenv("PG_HOST")
PG_PORT = os.getenv("PG_PORT")
PG_DATABASE = os.getenv("PG_DATABASE")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")

GRAPH_NAME = "payer_policy_knowledge_graph"

client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint=os.getenv(
        "AZURE_OPENAI_ENDPOINT",
        "https://ciaiciath2-foundry-dev.cognitiveservices.azure.com/",
    ),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)

models = ["gpt-5.6-luna", "gpt-5.4-mini"]
NL2CYPHER_MODEL = models[0]
ANSWER_MODEL = models[0]
MAX_QUERY_ATTEMPTS = 3
MAX_RESULT_ROWS = 20

POLICY_RESULT_COLUMNS = [
    "document_id",
    "field_name",
    "value",
    "evidence_quote",
    "page_number",
    "section_title",
    "source_file",
    "source_url",
]

In [2]:
def connect_postgres():
    return psycopg.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DATABASE,
        user=PG_USER,
        password=PG_PASSWORD,
        connect_timeout=10,
    )


def init_age(cursor):
    cursor.execute("LOAD 'age';")
    cursor.execute('SET search_path = ag_catalog, "$user", public;')


def open_age_connection():
    conn = connect_postgres()
    try:
        with conn.cursor() as cursor:
            init_age(cursor)
        conn.commit()
        return conn
    except Exception:
        conn.rollback()
        conn.close()
        raise


def normalize_agtype(value):
    if value is None or isinstance(value, (list, dict, int, float, bool)):
        return value
    text = str(value).strip()
    try:
        return json.loads(text)
    except Exception:
        return text.strip('"')


test_conn = open_age_connection()
test_conn.close()
print("Connected to PostgreSQL AGE graph:", GRAPH_NAME)

Connected to PostgreSQL AGE graph: payer_policy_knowledge_graph


In [3]:
def get_age_graph_schema(graph_name):
    nodes = {}
    relationships = []

    conn = open_age_connection()
    try:
        with conn.cursor() as cursor:
            cursor.execute(f"""
            SELECT *
            FROM cypher('{graph_name}', $$
                MATCH (n)
                RETURN DISTINCT labels(n), keys(n)
            $$) AS (labels agtype, properties agtype);
            """)
            for labels_value, properties_value in cursor.fetchall():
                labels = normalize_agtype(labels_value)
                properties = normalize_agtype(properties_value)
                labels = labels if isinstance(labels, list) else [labels]
                properties = properties if isinstance(properties, list) else [properties]
                for label in labels:
                    nodes.setdefault(str(label), set()).update(str(p) for p in properties)

            cursor.execute(f"""
            SELECT *
            FROM cypher('{graph_name}', $$
                MATCH (a)-[r]->(b)
                RETURN DISTINCT labels(a), type(r), labels(b)
            $$) AS (source_labels agtype, relationship agtype, target_labels agtype);
            """)
            for source, relationship, target in cursor.fetchall():
                relationships.append({
                    "source": normalize_agtype(source),
                    "relationship": normalize_agtype(relationship),
                    "target": normalize_agtype(target),
                })
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

    return {
        "nodes": {label: sorted(props) for label, props in nodes.items()},
        "relationships": relationships,
    }


def build_schema_text(schema):
    lines = ["NODE LABELS AND PROPERTIES"]
    for label, properties in schema["nodes"].items():
        lines.append(f"{label}: {', '.join(properties)}")
    lines.append("")
    lines.append("RELATIONSHIPS")
    for relation in schema["relationships"]:
        source = relation["source"]
        target = relation["target"]
        source = ", ".join(source) if isinstance(source, list) else source
        target = ", ".join(target) if isinstance(target, list) else target
        lines.append(f"({source})-[:{relation['relationship']}]->({target})")
    return "\n".join(lines)


age_schema = get_age_graph_schema(GRAPH_NAME)
GRAPH_SCHEMA = build_schema_text(age_schema)
print(GRAPH_SCHEMA)

NODE LABELS AND PROPERTIES
State: name, node_id, search_text
Criterion: authoritative, document_id, evidence_chunk_id, evidence_quote, field_name, node_id, page_number, quote_verified, reconciled_from_review, search_text, section_title, source_file, source_url, subject, validation_status, value
Payer: aliases, evidence_quote, name, node_id, search_text, source_document_id
CriterionType: field_name, node_id, routing_cues, search_text
Corpus: content_version, node_id, search_text, subject, version
EvidenceChunk: chunk_id, content, document_id, node_id, page_number, part_index, search_text, section_title, source_file, source_url, text
LineOfBusiness: name, node_id, search_text
PolicyDocument: aliases, conversion_status, doc_blob_url, document_id, document_type, node_id, search_text, source_file, source_hash, source_url, subject_relevant
Drug: name, node_id, search_text
Section: document_id, node_id, page_number, search_text, section_title, source_file

RELATIONSHIPS
(Corpus)-[:HAS_DOCUMEN

In [4]:
FORBIDDEN_CYPHER = [
    "CREATE", "MERGE", "DELETE", "DETACH", "SET", "REMOVE",
    "DROP", "LOAD CSV", "FOREACH", "CALL",
]


def strip_string_literals(cypher):
    output = []
    quote = None
    escaped = False
    for char in cypher:
        if quote:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == quote:
                quote = None
                output.append("''")
            continue
        if char in {"'", '"'}:
            quote = char
        else:
            output.append(char)
    return "".join(output)


def validate_column_name(name):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Invalid result column: {name}")
    return name


def validate_query_spec(query_spec):
    if not isinstance(query_spec, dict):
        raise ValueError("Query specification must be a JSON object")

    cypher = str(query_spec.get("cypher", "")).strip()
    columns = query_spec.get("columns")
    if not cypher or not isinstance(columns, list) or not columns:
        raise ValueError("Query specification requires cypher and columns")

    columns = [validate_column_name(str(column)) for column in columns]
    literal_free = strip_string_literals(cypher)
    normalized = literal_free.upper().strip()

    if ";" in normalized:
        raise ValueError("Semicolons are not allowed")
    if not re.match(r"^(MATCH|WITH|UNWIND)\b", normalized):
        raise ValueError("Query must start with MATCH, WITH, or UNWIND")
    for keyword in FORBIDDEN_CYPHER:
        if re.search(rf"\b{re.escape(keyword)}\b", normalized):
            raise ValueError(f"Unsafe Cypher keyword: {keyword}")
    if re.search(r"\bOPTIONAL\s+MATCH\b", normalized):
        raise ValueError("OPTIONAL MATCH is disabled for Apache AGE compatibility")
    if re.search(r"\bTOSTRING\s*\(", normalized):
        raise ValueError("toString() is disabled for Apache AGE compatibility")
    if re.search(r"\bTOLOWER\s*\((?!\s*[A-Za-z_]\w*\.search_text\s*\))", normalized):
        raise ValueError("toLower() is permitted only for scalar search_text")
    if len(columns) > 20:
        raise ValueError("Too many result columns")

    return {"cypher": cypher, "columns": columns}

In [5]:
def llm_json(messages, model):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


RETRIEVAL_PLAN_SYSTEM_PROMPT = """
Create a retrieval plan for a payer-policy knowledge graph.
Return JSON only with this shape:
{
  "intent": "single_policy|comparison|intersection_count|stringency_ranking",
  "field_names": ["live_field_name"],
  "document_terms": ["payer or document alias"],
  "line_of_business": "Commercial|Medicaid|null",
  "evidence_terms": ["exact source-text phrase"]
}

Rules:
- Select field_names only from the supplied live criterion types.
- Complete PA criteria or stringency needs all criterion types; return field_names [].
- Multiple-condition plan counts use intersection_count.
- A typical value, range, or cross-plan question uses comparison.
- document_terms contains only explicitly named payer/document aliases.
- Do not answer and do not generate Cypher.
""".strip()


def request_retrieval_plan(question, field_names, document_catalog):
    compact_catalog = [
        {
            "document_id": item["document_id"],
            "aliases": item.get("aliases", []),
            "payer": item.get("payer"),
            "line_of_business": item.get("line_of_business"),
        }
        for item in document_catalog
    ]
    return llm_json(
        [
            {"role": "system", "content": RETRIEVAL_PLAN_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"QUESTION:\n{question}\n\n"
                    f"LIVE CRITERION TYPES:\n{json.dumps(field_names)}\n\n"
                    f"DOCUMENT CATALOG:\n"
                    f"{json.dumps(compact_catalog, ensure_ascii=False)}"
                ),
            },
        ],
        NL2CYPHER_MODEL,
    )

In [6]:
def execute_policy_query(query_spec):
    query_spec = validate_query_spec(query_spec)
    columns = query_spec["columns"]
    column_definition = ", ".join(f"{column} agtype" for column in columns)
    sql = f"""
    SELECT *
    FROM cypher('{GRAPH_NAME}', $$
        {query_spec["cypher"]}
    $$) AS ({column_definition});
    """

    conn = open_age_connection()
    try:
        with conn.cursor() as cursor:
            cursor.execute(sql)
            rows = cursor.fetchall()
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

    return [
        {column: normalize_agtype(value) for column, value in zip(columns, row)}
        for row in rows
    ]

In [7]:
POLICY_SNAPSHOT_QUERIES = {
    "documents": {
        "cypher": """MATCH (d:PolicyDocument)
RETURN d.document_id AS document_id, d.source_file AS source_file,
       d.source_url AS source_url, d.doc_blob_url AS doc_blob_url,
       d.aliases AS aliases, d.search_text AS search_text
LIMIT 500""",
        "columns": ["document_id", "source_file", "source_url", "doc_blob_url", "aliases", "search_text"],
    },
    "payers": {
        "cypher": """MATCH (d:PolicyDocument)-[:ISSUED_BY]->(p:Payer)
RETURN d.document_id AS document_id, p.name AS payer, p.aliases AS payer_aliases
LIMIT 500""",
        "columns": ["document_id", "payer", "payer_aliases"],
    },
    "lines_of_business": {
        "cypher": """MATCH (d:PolicyDocument)-[r:HAS_LINE_OF_BUSINESS]->(lob:LineOfBusiness)
RETURN d.document_id AS document_id, lob.name AS line_of_business,
       r.evidence_quote AS lob_evidence_quote
LIMIT 500""",
        "columns": ["document_id", "line_of_business", "lob_evidence_quote"],
    },
    "criterion_types": {
        "cypher": """MATCH (t:CriterionType)
RETURN t.field_name AS field_name, t.routing_cues AS routing_cues
LIMIT 500""",
        "columns": ["field_name", "routing_cues"],
    },
    "criteria": {
        "cypher": """MATCH (d:PolicyDocument)-[:HAS_CRITERION]->(c:Criterion)-[:OF_TYPE]->(t:CriterionType)
RETURN d.document_id AS document_id, c.subject AS subject, t.field_name AS field_name,
       c.value AS value, c.evidence_quote AS evidence_quote,
       c.page_number AS page_number, c.section_title AS section_title,
       c.source_file AS source_file, c.source_url AS source_url,
       c.confidence AS confidence
LIMIT 1000""",
        "columns": ["document_id", "subject", "field_name", "value", "evidence_quote", "page_number", "section_title", "source_file", "source_url", "confidence"],
    },
    "evidence_chunks": {
        "cypher": """MATCH (d:PolicyDocument)-[:HAS_SECTION]->(s:Section)-[:HAS_EVIDENCE]->(e:EvidenceChunk)
RETURN d.document_id AS document_id, e.chunk_id AS chunk_id,
       e.content AS content, e.page_number AS page_number,
       e.section_title AS section_title, e.source_file AS source_file
LIMIT 2000""",
        "columns": ["document_id", "chunk_id", "content", "page_number", "section_title", "source_file"],
    },
}

_POLICY_SNAPSHOT_CACHE = None


def normalize_match_text(value):
    text = str(value or "").casefold()
    text = text.replace("≥", " greater than or equal to ")
    text = text.replace(">=", " greater than or equal to ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def list_value(value):
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def infer_line_of_business(document):
    explicit = normalize_match_text(document.get("line_of_business"))
    if "medicaid" in explicit:
        return "Medicaid", "explicit graph identity"
    if "commercial" in explicit:
        return "Commercial", "explicit graph identity"

    material = normalize_match_text(" ".join(
        str(value) for value in [
            document.get("document_id"), document.get("source_file"),
            document.get("source_url"), document.get("payer"),
            *list_value(document.get("aliases")),
            *list_value(document.get("payer_aliases")),
        ] if value
    ))
    # Medicaid evidence wins over a misleading filename such as Aetna's.
    if any(marker in material for marker in (
        "medicaid", "badgercare", "forwardhealth", "aetna better health", "nhpri"
    )):
        return "Medicaid", "document identifier, alias, payer, or source URL"
    if "commercial" in material:
        return "Commercial", "document identifier, alias, payer, or source URL"
    return "Unclassified", "no explicit commercial or Medicaid evidence"


def load_policy_snapshot(force=False):
    global _POLICY_SNAPSHOT_CACHE
    if _POLICY_SNAPSHOT_CACHE is not None and not force:
        return _POLICY_SNAPSHOT_CACHE

    raw = {
        name: execute_policy_query(spec)
        for name, spec in POLICY_SNAPSHOT_QUERIES.items()
    }
    documents = {
        row["document_id"]: {**row}
        for row in raw["documents"] if row.get("document_id")
    }
    for row in raw["payers"]:
        if row.get("document_id") in documents:
            documents[row["document_id"]].update({
                "payer": row.get("payer"),
                "payer_aliases": row.get("payer_aliases") or [],
            })
    for row in raw["lines_of_business"]:
        if row.get("document_id") in documents:
            documents[row["document_id"]]["line_of_business"] = row.get("line_of_business")
            documents[row["document_id"]]["lob_evidence_quote"] = row.get("lob_evidence_quote")

    for document in documents.values():
        classification, basis = infer_line_of_business(document)
        document["retrieval_line_of_business"] = classification
        document["classification_basis"] = basis

    _POLICY_SNAPSHOT_CACHE = {
        "documents": documents,
        "criterion_types": {
            row["field_name"]: list_value(row.get("routing_cues"))
            for row in raw["criterion_types"] if row.get("field_name")
        },
        "criteria": raw["criteria"],
        "evidence_chunks": raw["evidence_chunks"],
    }
    return _POLICY_SNAPSHOT_CACHE


def document_match_material(document):
    return normalize_match_text(" ".join(
        str(value) for value in [
            document.get("document_id"), document.get("source_file"),
            document.get("source_url"), document.get("payer"),
            *list_value(document.get("aliases")),
            *list_value(document.get("payer_aliases")),
        ] if value
    ))


def contains_phrase(text, phrase):
    if not phrase:
        return False
    if " " not in phrase and len(phrase) <= 4:
        return re.search(rf"\b{re.escape(phrase)}\b", text) is not None
    return phrase in text


def named_document_ids(question, documents, requested_terms=None):
    question_text = normalize_match_text(question)
    requested_terms = requested_terms or []
    matches = set()
    for document_id, document in documents.items():
        material = document_match_material(document)
        aliases = [
            document_id, document.get("payer"),
            *list_value(document.get("aliases")),
            *list_value(document.get("payer_aliases")),
            *requested_terms,
        ]
        for alias in aliases:
            alias_text = normalize_match_text(alias)
            if (
                len(alias_text) >= 3
                and contains_phrase(question_text, alias_text)
                and contains_phrase(material, alias_text)
            ):
                matches.add(document_id)
                break
    return matches


def build_verified_retrieval_plan(question, snapshot):
    live_fields = set(snapshot["criterion_types"])
    documents = list(snapshot["documents"].values())
    try:
        proposed = request_retrieval_plan(question, sorted(live_fields), documents)
    except Exception as error:
        proposed = {"planner_error": f"{type(error).__name__}: {error}"}

    q = normalize_match_text(question)
    intent = proposed.get("intent", "single_policy")
    if re.search(r"\b(how many|count|number of)\b", q):
        intent = "intersection_count"
    elif "most stringent" in q or "stringency" in q or ("rank" in q and "criteria" in q):
        intent = "stringency_ranking"
    elif re.search(r"\b(typical|range|compare|across)\b", q):
        intent = "comparison"
    if intent not in {"single_policy", "comparison", "intersection_count", "stringency_ranking"}:
        intent = "single_policy"

    fields = [field for field in proposed.get("field_names", []) if field in live_fields]
    if "upcr" in q or "urine protein to creatinine" in q:
        fields = ["upcr"]
    if "egfr" in q:
        fields.append("egfr")
    if ("ace" in q and "arb" in q) or "ras inhibitor" in q or "raas inhibitor" in q:
        fields.extend(["ace_arb_requirement", "ace_arb_duration", "ace_arb_exception"])
    if intent == "stringency_ranking" or (intent == "single_policy" and "criteria" in q):
        fields = []
    fields = list(OrderedDict.fromkeys(field for field in fields if field in live_fields))

    requested_lob = proposed.get("line_of_business")
    if "medicaid" in q or "medciad" in q:
        requested_lob = "Medicaid"
    elif "commercial" in q:
        requested_lob = "Commercial"
    if requested_lob not in {"Medicaid", "Commercial"}:
        requested_lob = None

    named_ids = named_document_ids(question, snapshot["documents"], proposed.get("document_terms", []))
    if named_ids and intent == "single_policy":
        document_ids = named_ids
    elif requested_lob == "Medicaid":
        document_ids = {
            key for key, item in snapshot["documents"].items()
            if item["retrieval_line_of_business"] == "Medicaid"
        }
    elif requested_lob == "Commercial":
        # Retain Unclassified plans to show a separate broader/non-Medicaid result.
        document_ids = {
            key for key, item in snapshot["documents"].items()
            if item["retrieval_line_of_business"] in {"Commercial", "Unclassified"}
        }
    else:
        document_ids = set(snapshot["documents"])

    terms = [
        normalize_match_text(term) for term in proposed.get("evidence_terms", [])
        if normalize_match_text(term)
    ]
    if "upcr" in fields:
        terms.extend(["upcr", "urine protein to creatinine", "proteinuria"])
    if "egfr" in fields:
        terms.extend(["egfr", "glomerular filtration"])
    if any(field.startswith("ace_arb") for field in fields):
        terms.extend(["ace inhibitor", "arb", "ras inhibitor", "raas inhibitor"])
    if intent == "stringency_ranking":
        terms.extend(["all of the following", "approval criteria", "coverage criteria", "must meet", "and"])
    if intent == "single_policy" and not fields:
        terms.extend(["initial", "approval", "criteria", "reauthorization", "renewal"])

    return {
        "intent": intent,
        "field_names": fields,
        "document_ids": sorted(document_ids),
        "named_document_ids": sorted(named_ids),
        "line_of_business": requested_lob,
        "evidence_terms": list(OrderedDict.fromkeys(terms)),
        "planner_output": proposed,
    }


NON_INITIAL_OR_ADMIN_FIELDS = {
    "documentation", "procedural_requirements", "initial_authorization_duration",
    "reauthorization_duration", "reauthorization_requirements",
    "automated_authorization_pathway",
}


def select_relevant_chunks(snapshot, document_ids, terms, max_per_document=6):
    selected = []
    for document_id in document_ids:
        scored = []
        for row in snapshot["evidence_chunks"]:
            if row.get("document_id") != document_id:
                continue
            content = normalize_match_text(row.get("content"))
            score = sum(
                4 if " " in term and term in content else 1
                for term in terms if term and term in content
            )
            if score:
                scored.append((score, len(content), row))
        scored.sort(key=lambda item: (-item[0], item[1]))
        selected.extend(row for _, _, row in scored[:max_per_document])
    return selected


def build_computed_facts(plan, documents, criteria):
    by_document = {document_id: [] for document_id in plan["document_ids"]}
    for row in criteria:
        by_document.setdefault(row["document_id"], []).append(row)

    coverage = {}
    for document_id in plan["document_ids"]:
        rows = by_document.get(document_id, [])
        fields = {row.get("field_name") for row in rows}
        initial_rows = [
            row for row in rows
            if row.get("field_name") not in NON_INITIAL_OR_ADMIN_FIELDS
        ]
        coverage[document_id] = {
            "retrieved_fields": sorted(field for field in fields if field),
            "retrieved_row_count": len(rows),
            "initial_clinical_row_count_proxy": len(initial_rows),
            "initial_clinical_field_count_proxy": len({
                row.get("field_name") for row in initial_rows if row.get("field_name")
            }),
            "classification": documents[document_id]["retrieval_line_of_business"],
        }

    facts = {
        "document_coverage": coverage,
        "counting_warning": (
            "Criterion rows are extracted facts, not a stored Boolean expression. "
            "Counts are proxies; inspect evidence chunks for nested AND/OR logic."
        ),
    }
    requested = set(plan["field_names"])
    if "egfr" in requested and "ace_arb_requirement" in requested:
        qualifying = [
            document_id for document_id, item in coverage.items()
            if {"egfr", "ace_arb_requirement"}.issubset(set(item["retrieved_fields"]))
        ]
        facts["egfr_and_ace_arb_documents"] = qualifying
        facts["strict_commercial_matches"] = [
            document_id for document_id in qualifying
            if coverage[document_id]["classification"] == "Commercial"
        ]
        facts["unclassified_additional_matches"] = [
            document_id for document_id in qualifying
            if coverage[document_id]["classification"] == "Unclassified"
        ]
    return facts


def retrieve_policy_evidence(question, force_refresh=False):
    snapshot = load_policy_snapshot(force=force_refresh)
    plan = build_verified_retrieval_plan(question, snapshot)
    document_ids = set(plan["document_ids"])
    fields = set(plan["field_names"])

    criteria = [
        row for row in snapshot["criteria"]
        if row.get("document_id") in document_ids
        and (not fields or row.get("field_name") in fields)
    ]
    chunks = select_relevant_chunks(snapshot, plan["document_ids"], plan["evidence_terms"])
    documents = {
        key: snapshot["documents"][key] for key in plan["document_ids"]
    }
    computed = build_computed_facts(plan, documents, criteria)

    grouped = [
        {
            "document": documents[document_id],
            "criteria": [row for row in criteria if row.get("document_id") == document_id],
            "evidence_chunks": [row for row in chunks if row.get("document_id") == document_id],
            "coverage": computed["document_coverage"][document_id],
        }
        for document_id in plan["document_ids"]
    ]
    return {"plan": plan, "computed_facts": computed, "documents": grouped}

In [8]:
POLICY_ANSWER_SYSTEM_PROMPT = """
Answer payer-policy questions only from the verified graph retrieval package.

Evidence rules:
- criterion.value and criterion.evidence_quote are primary facts.
- Evidence chunks can fill an extraction gap, but label overview or trial text as
  contextual rather than an operative PA requirement.
- Cite material claims as [document_id, page N, section title].
- Never silently classify an Unclassified plan as Commercial or Medicaid.
- For Commercial questions, give the strict Commercial result first, then report
  relevant Unclassified plans separately as a broader/non-Medicaid interpretation.
- For ranges, inspect every selected plan, state the observed minimum-to-maximum,
  list each threshold, and mention documents without an operative threshold.
- For intersections, count distinct documents with every requested primary field.
- For stringency, never count occurrences of the word AND. Criterion row/field
  counts are only proxies because the graph lacks a complete Boolean tree. Use
  source chunks to explain nested logic and avoid unsupported exact rankings.
- For one payer's PA criteria, list all initial requirements and separately list
  reauthorization requirements.
- Do not add outside medical or policy knowledge.
""".strip()


def generate_policy_answer(question, retrieval):
    if not any(item["criteria"] or item["evidence_chunks"] for item in retrieval["documents"]):
        return "No matching verified policy evidence was found."

    response = client.chat.completions.create(
        model=ANSWER_MODEL,
        messages=[
            {"role": "system", "content": POLICY_ANSWER_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"QUESTION:\n{question}\n\n"
                    f"VERIFIED RETRIEVAL PACKAGE:\n"
                    f"{json.dumps(retrieval, ensure_ascii=False, default=str)}"
                ),
            },
        ],
    )
    return response.choices[0].message.content


def ask_policy_graph(question, show_cypher=True, show_raw_result=False, force_refresh=False):
    retrieval = retrieve_policy_evidence(question, force_refresh=force_refresh)
    answer = generate_policy_answer(question, retrieval)
    executed_queries = {
        name: spec["cypher"] for name, spec in POLICY_SNAPSHOT_QUERIES.items()
    }
    result = {
        "question": question,
        "answer": answer,
        "cypher_query": executed_queries,
        "structured_response": retrieval,
        "strategy": "verified_snapshot_retrieval",
    }

    if show_cypher:
        print("Retrieval strategy: verified snapshot queries (no LLM-generated Cypher execution)")
        print(json.dumps(executed_queries, indent=2))
    if show_raw_result:
        print("Retrieval package:\n", json.dumps(
            retrieval, indent=2, ensure_ascii=False, default=str
        ))
    return result


def refresh_policy_retrieval_cache():
    return load_policy_snapshot(force=True)

## Production guardrails

This stage validates source scope, unsupported dimensions, identity provenance, citations, negative claims, and cross-document completeness before an answer is returned.

In [ ]:
PRODUCTION_GUARDRAIL_VERSION = "1.0"
if "_unguarded_retrieve_policy_evidence" not in globals():
    _unguarded_retrieve_policy_evidence = retrieve_policy_evidence
if "_unguarded_generate_policy_answer" not in globals():
    _unguarded_generate_policy_answer = generate_policy_answer


def graph_has_dimension(*terms):
    schema_text = normalize_match_text(json.dumps(age_schema, default=str))
    return any(normalize_match_text(term) in schema_text for term in terms)


def question_tokens_by_rarity(question, snapshot):
    query_text = normalize_match_text(question)
    query_text = re.sub(
        r"^(?:what|which|does|do|for)\b(?:\s+(?:policy|policies|documents?))?\s*",
        "",
        query_text,
    )
    query_text = re.sub(
        r"\b(?:cover|covers|coverage rules apply to|have|has|give|rank|among them)\b",
        " ",
        query_text,
    )
    tokens = set(re.findall(r"[a-z0-9]+", query_text))
    tokens = {token for token in tokens if len(token) >= 4}
    if not tokens:
        return []
    document_frequency = {token: set() for token in tokens}
    for row in snapshot["evidence_chunks"]:
        content = normalize_match_text(row.get("content"))
        for token in tokens:
            if re.search(rf"\b{re.escape(token)}\b", content):
                document_frequency[token].add(row.get("document_id"))
    maximum_frequency = max(2, len(snapshot["documents"]) // 3)
    return sorted(
        [token for token, ids in document_frequency.items() if ids and len(ids) <= maximum_frequency],
        key=lambda token: (len(document_frequency[token]), -len(token), token),
    )


def printed_page_before_term(content, term):
    content = str(content or "")
    position = content.casefold().find(term.casefold())
    if position < 0:
        return None
    pages = re.findall(r"\bPage\s+(\d+)\b", content[:position], flags=re.IGNORECASE)
    return int(pages[-1]) if pages else None


def rare_source_matches(question, snapshot, document_ids, limit=30):
    rare_terms = question_tokens_by_rarity(question, snapshot)
    primary_term = rare_terms[0] if rare_terms else None
    matches = []
    for row in snapshot["evidence_chunks"]:
        if row.get("document_id") not in document_ids:
            continue
        content = str(row.get("content") or "")
        normalized = normalize_match_text(content)
        matched = [
            term for term in rare_terms
            if re.search(rf"\b{re.escape(term)}\b", normalized)
        ]
        if not matched or primary_term not in matched:
            continue
        positions = [normalized.find(term) for term in matched if normalized.find(term) >= 0]
        position = min(positions) if positions else 0
        start = max(0, position - 250)
        excerpt = re.sub(r"\s+", " ", content[start:position + 900]).strip()
        matches.append({
            "document_id": row.get("document_id"),
            "matched_terms": matched,
            "excerpt": excerpt,
            "page_number": row.get("page_number"),
            "printed_page": printed_page_before_term(content, matched[0]),
            "section_title": row.get("section_title"),
            "source_file": row.get("source_file"),
            "chunk_id": row.get("chunk_id"),
        })
    matches.sort(key=lambda item: (-len(item["matched_terms"]), item["document_id"]))
    return matches[:limit]


def body_verified_medicaid_documents(snapshot):
    verified = []
    for document_id, document in snapshot["documents"].items():
        evidence = normalize_match_text(document.get("lob_evidence_quote"))
        document_id_text = normalize_match_text(document_id)
        source_stem = normalize_match_text(
            str(document.get("source_file") or "").rsplit(".", 1)[0]
        )
        valid_body_evidence = bool(
            "medicaid" in evidence
            and evidence not in {document_id_text, source_stem}
        )
        reviewed_aliases = normalize_match_text(" ".join(
            str(value) for value in [
                *list_value(document.get("aliases")),
                *list_value(document.get("payer_aliases")),
                document.get("payer"),
            ] if value
        ))
        if valid_body_evidence or "medicaid" in reviewed_aliases:
            verified.append(document_id)
    return sorted(verified)


def build_guardrail_context(question, retrieval, snapshot, source_matches):
    q = normalize_match_text(question)
    asks_lives = bool(re.search(
        r"\b(covered lives|current lives|most lives|enrollment|membership|lives map|lives footprint)\b", q
    ))
    asks_plan_count = bool(re.search(
        r"\b(plan count|number of plans|largest number of plans)\b", q
    ))
    explicit_scope = re.search(
        r"(?:coverage rules apply to|policies cover)\s+([a-z0-9-]+)", q
    )
    scope_term = explicit_scope.group(1) if explicit_scope else None
    primary_subjects = sorted({
        normalize_match_text(row.get("subject"))
        for item in retrieval["documents"] for row in item["criteria"]
        if row.get("subject")
    })
    source_only_scope = bool(
        scope_term and scope_term not in primary_subjects
        and any(scope_term in item["matched_terms"] for item in source_matches)
    )
    matched_documents = sorted({
        item["document"]["document_id"]
        for item in retrieval["documents"] if item["criteria"] or item["evidence_chunks"]
    })
    return {
        "version": PRODUCTION_GUARDRAIL_VERSION,
        "covered_lives_requested": asks_lives,
        "covered_lives_supported": graph_has_dimension(
            "covered_lives", "covered lives", "enrollment", "membership"
        ),
        "plan_count_requested": asks_plan_count,
        "state_plan_dimension_supported": graph_has_dimension(
            "plan_count", "plan count", "state_plan", "state plan"
        ),
        "body_verified_medicaid_documents": body_verified_medicaid_documents(snapshot),
        "body_verified_lob_only": "establish medicaid lob" in q,
        "negative_pa_question": "do not require prior authorization" in q,
        "silence_question": "silent on" in q,
        "source_only_scope": source_only_scope,
        "explicit_source_scope_term": scope_term,
        "primary_criterion_subjects": primary_subjects,
        "rare_source_matches": source_matches,
        "matching_policy_document_ids": matched_documents,
        "policy_document_count_is_not_plan_count": True,
    }


def retrieve_policy_evidence(question, force_refresh=False):
    retrieval = _unguarded_retrieve_policy_evidence(
        question, force_refresh=force_refresh
    )
    snapshot = load_policy_snapshot(force=False)
    document_ids = set(retrieval["plan"]["document_ids"])
    source_matches = rare_source_matches(question, snapshot, document_ids)

    matches_by_document = {}
    for match in source_matches:
        matches_by_document.setdefault(match["document_id"], []).append(match)
    for item in retrieval["documents"]:
        existing_ids = {row.get("chunk_id") for row in item["evidence_chunks"]}
        for match in matches_by_document.get(item["document"]["document_id"], []):
            if match.get("chunk_id") not in existing_ids:
                item["evidence_chunks"].append(match)

    retrieval["guardrails"] = build_guardrail_context(
        question, retrieval, snapshot, source_matches
    )
    return retrieval


ANSWER_GUARDRAIL_PROMPT = """
Audit and, when necessary, rewrite a draft policy answer. Return JSON only:
{"answer": "final grounded answer", "issues_corrected": []}

Mandatory rules:
- Use only the retrieval package; every policy claim needs a valid document citation.
- If covered lives/enrollment is requested but unsupported, state it is unavailable.
  Never substitute policy-document counts, payer counts, or a proxy ranking.
- If plan count is requested without a state-plan dimension, mark plan count unavailable.
  A policy document is not a plan.
- For body-verified Medicaid LOB, use exactly body_verified_medicaid_documents; do not
  promote filename/URL inference to body-verified identity.
- source_only_scope means criterion rows belong to another product. Use only matching
  source excerpts for the requested product and exclude unrelated product criteria.
- A rare_source_match is positive evidence; do not answer zero/none when it directly
  satisfies the question. Prefer printed_page when supplied.
- Silence means no verified operative field in this corpus, not proof that a policy
  does not require it. Never infer no prior authorization from missing evidence.
- Threshold intersections require the correct operator, numeric value, and unit.
- Do not count occurrences of AND or claim exact restrictiveness from proxy counts.
- Lists described as exhaustive must inspect every selected document.
- Keep counts consistent with the listed distinct documents.
""".strip()


def safe_guardrail_fallback(question, retrieval):
    guard = retrieval["guardrails"]
    matches = guard["matching_policy_document_ids"]
    if guard["covered_lives_requested"] and not guard["covered_lives_supported"]:
        return (
            "Covered-lives ranking is unavailable because this graph contains no "
            "verified enrollment or covered-lives dimension. Matching policy documents: "
            + (", ".join(matches) if matches else "none verified")
            + ". Policy-document counts are not a substitute for lives."
        )
    if guard["plan_count_requested"] and not guard["state_plan_dimension_supported"]:
        return (
            "Plan count is unavailable because the graph has no state-plan dimension. "
            "A retrieved policy document must not be counted as a plan. Matching policy "
            "documents: " + (", ".join(matches) if matches else "none verified") + "."
        )
    if guard["rare_source_matches"]:
        lines = ["Verified source matches:"]
        for match in guard["rare_source_matches"][:10]:
            page = match.get("printed_page") or match.get("page_number")
            lines.append(
                f"- {match['document_id']}, page {page}: {match['excerpt']}"
            )
        return "\n".join(lines)
    return "No answer passed the production grounding guardrails."


def validate_final_answer(question, answer, retrieval):
    guard = retrieval["guardrails"]
    allowed_ids = {item["document"]["document_id"] for item in retrieval["documents"]}
    cited_ids = set(re.findall(r"\[(\d{2}_[a-z0-9_]+)", answer.casefold()))
    if cited_ids - allowed_ids:
        return False, f"Citations outside retrieval: {sorted(cited_ids - allowed_ids)}"
    normalized = normalize_match_text(answer)
    if guard["covered_lives_requested"] and not guard["covered_lives_supported"]:
        if not any(term in normalized for term in ("unavailable", "cannot determine", "does not contain")):
            return False, "Missing covered-lives unavailability disclosure"
        if "using the number of" in normalized and "proxy" in normalized:
            return False, "Unsupported lives proxy"
    if guard["plan_count_requested"] and not guard["state_plan_dimension_supported"]:
        if re.search(r"plan count\s*[:=-]?\s*\d+", normalized):
            return False, "Policy-document count presented as plan count"
    if guard["body_verified_lob_only"]:
        forbidden = {"12_aetna_commercial", "14_vchcp_ca_medicaid"}
        if forbidden.intersection(set(re.findall(r"\b\d{2}_[a-z0-9_]+\b", normalized))):
            return False, "Inferred LOB presented as body verified"
    if guard["rare_source_matches"] and (
        normalized.startswith("none") or "no policy in the retrieved" in normalized
    ):
        return False, "Positive source evidence contradicted"
    return True, None


def generate_policy_answer(question, retrieval):
    draft = _unguarded_generate_policy_answer(question, retrieval)
    try:
        audited = llm_json([
            {"role": "system", "content": ANSWER_GUARDRAIL_PROMPT},
            {"role": "user", "content": (
                f"QUESTION:\n{question}\n\nDRAFT:\n{draft}\n\n"
                f"RETRIEVAL PACKAGE:\n{json.dumps(retrieval, ensure_ascii=False, default=str)}"
            )},
        ], ANSWER_MODEL)
        answer = str(audited.get("answer") or "").strip()
    except Exception:
        answer = draft
    passed, _ = validate_final_answer(question, answer, retrieval)
    return answer if passed else safe_guardrail_fallback(question, retrieval)


## Deterministic content check

Run this before asking natural-language questions. It confirms that the graph contains extracted criterion values and source evidence, independently of the LLM.

In [9]:
content_check = execute_policy_query({
    "cypher": """
MATCH (d:PolicyDocument)-[:HAS_CRITERION]->(c:Criterion)-[:OF_TYPE]->(t:CriterionType)
RETURN d.document_id AS document_id,
       t.field_name AS field_name,
       c.value AS value,
       c.evidence_quote AS evidence_quote,
       c.page_number AS page_number,
       c.section_title AS section_title,
       c.source_file AS source_file,
       c.source_url AS source_url
LIMIT 5
""".strip(),
    "columns": POLICY_RESULT_COLUMNS,
})
print(json.dumps(content_check, indent=2, ensure_ascii=False, default=str))

[
  {
    "document_id": "01_wi_medicaid",
    "field_name": "procedural_requirements",
    "value": "Voyxact must be prescribed in a dose and manner consistent with FDA-approved product labeling.",
    "evidence_quote": "Voyxact must be prescribed in a dose and manner consistent with FDA -approved product labeling.",
    "page_number": 1,
    "section_title": "Clinical Criteria for Voyxact",
    "source_file": "01_wi_medicaid.html",
    "source_url": "https://www.forwardhealth.wi.gov/WIPortal/Subsystem/KW/Print.aspx?ia=1&p=1&sa=48&s=3&c=11&nt=Voyxact&adv=Y"
  },
  {
    "document_id": "01_wi_medicaid",
    "field_name": "procedural_requirements",
    "value": "Voyxact PA requests must be completed, signed, and dated by the prescriber.",
    "evidence_quote": "PA requests for Voyxact must be completed, signed, and dated by the prescriber.",
    "page_number": 1,
    "section_title": "Document",
    "source_file": "01_wi_medicaid.html",
    "source_url": "https://www.forwardhealth.wi.go

## Example policy questions

The returned object always includes answer, cypher_query, structured_response, strategy, and all retry attempts.

In [10]:
def ask_policy_questions(
    questions,
    output_csv="policy_question_answers.csv",
    force_refresh=True,
    max_questions=50,
):
    """Return and save records containing only question and answer."""
    import csv
    import tempfile
    from pathlib import Path

    if not isinstance(questions, (list, tuple)):
        raise TypeError("questions must be a list or tuple of strings")
    if not questions:
        raise ValueError("questions must not be empty")
    if len(questions) > max_questions:
        raise ValueError(f"A batch may contain at most {max_questions} questions")

    cleaned_questions = []
    for question in questions:
        if not isinstance(question, str) or not question.strip():
            raise ValueError("Every question must be a non-empty string")
        cleaned_questions.append(question.strip())
    if len(set(cleaned_questions)) != len(cleaned_questions):
        raise ValueError("Duplicate questions are not allowed in a production batch")

    if force_refresh:
        refresh_policy_retrieval_cache()

    records = []
    for question in cleaned_questions:
        result = ask_policy_graph(
            question,
            show_cypher=False,
            show_raw_result=False,
            force_refresh=False,
        )
        answer = result.get("answer")
        if not isinstance(answer, str) or not answer.strip():
            raise RuntimeError(f"No verified answer was produced for: {question}")
        records.append({"question": question, "answer": answer.strip()})

    output_path = Path(output_csv).resolve()
    workspace_root = Path.cwd().resolve()
    if output_path.suffix.casefold() != ".csv":
        raise ValueError("output_csv must use the .csv extension")
    if output_path != workspace_root and workspace_root not in output_path.parents:
        raise ValueError("output_csv must remain inside the current workspace")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    def csv_safe(value):
        text = str(value)
        return "'" + text if text.startswith(("=", "+", "-", "@")) else text

    temporary_name = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", newline="", encoding="utf-8-sig",
            dir=output_path.parent, suffix=".tmp", delete=False,
        ) as file:
            temporary_name = file.name
            writer = csv.DictWriter(file, fieldnames=["question", "answer"])
            writer.writeheader()
            writer.writerows([
                {"question": csv_safe(item["question"]), "answer": csv_safe(item["answer"])}
                for item in records
            ])
        Path(temporary_name).replace(output_path)
    finally:
        if temporary_name and Path(temporary_name).exists():
            Path(temporary_name).unlink()

    return records


In [11]:
questions = [
      "What is the PA criteria for UHC commercial for Voyxact?",
      "How many commercial plans need eGFR >= 30 along with Ace/ARB use for approving Voyxact?",
      "What is the typical Medicaid UPCR requirement?",
      "Which are the most stringent plans in terms of PA criteria for Voyxact?"
  ]

question_answers = ask_policy_questions(questions)
print(question_answers)


[{'question': 'What is the PA criteria for UHC commercial for Voyxact?', 'answer': '## UHC Commercial: Voyxact PA criteria\n\n### Initial authorization\nUHC requires **all** of the following:\n\n1. **Diagnosis:** Primary immunoglobulin A nephropathy (IgAN) confirmed by renal biopsy.\n2. Patient is **at risk of rapid disease progression**.\n3. Voyxact will be used to **reduce proteinuria**.\n4. **eGFR ≥30 mL/min/1.73 m²**.\n5. **Renin-angiotensin system therapy—one of the following:**\n   - The patient is on a stabilized dose and receiving concomitant therapy with a maximally tolerated ACE inhibitor or ARB; **or**\n   - The patient has a contraindication or intolerance to both ACE inhibitors and ARBs.\n6. **Glucocorticoid history:** Failure after a **30-day trial** of a glucocorticoid, or contraindication/intolerance to a glucocorticoid such as budesonide, methylprednisolone, or prednisone.\n7. Voyxact is prescribed by, or in consultation with, a **nephrologist**.\n\n[02_uhc_commercial,

In [12]:
from IPython.display import Markdown, display

for index,item in enumerate(question_answers, start=1):
    display(Markdown(f"### Question {index}: {item['question']}"))
    display(Markdown(f"**Answer:** {item['answer']}"))

### Question 1: What is the PA criteria for UHC commercial for Voyxact?

**Answer:** ## UHC Commercial: Voyxact PA criteria

### Initial authorization
UHC requires **all** of the following:

1. **Diagnosis:** Primary immunoglobulin A nephropathy (IgAN) confirmed by renal biopsy.
2. Patient is **at risk of rapid disease progression**.
3. Voyxact will be used to **reduce proteinuria**.
4. **eGFR ≥30 mL/min/1.73 m²**.
5. **Renin-angiotensin system therapy—one of the following:**
   - The patient is on a stabilized dose and receiving concomitant therapy with a maximally tolerated ACE inhibitor or ARB; **or**
   - The patient has a contraindication or intolerance to both ACE inhibitors and ARBs.
6. **Glucocorticoid history:** Failure after a **30-day trial** of a glucocorticoid, or contraindication/intolerance to a glucocorticoid such as budesonide, methylprednisolone, or prednisone.
7. Voyxact is prescribed by, or in consultation with, a **nephrologist**.

[02_uhc_commercial, page 1, “A. Initial Authorization”] [02_uhc_commercial, page 2, “-AND-”]

### Reauthorization
- Documentation of a **positive clinical response**, demonstrated by a **reduction in proteinuria**.

[02_uhc_commercial, page 2, “B. Reauthorization”]

### Authorization duration and additional rules
- **Initial authorization:** 12 months.
- **Reauthorization:** 12 months.
- UHC may also approve initial authorization or reauthorization based solely on prior claim/medication history, ICD-10 diagnosis codes, and/or claim logic. Supply limits may apply.

[02_uhc_commercial, page 2, “Authorization will be issued for 12 months”] [02_uhc_commercial, page 2, “3. Additional Clinical Rules:”]

### Question 2: How many commercial plans need eGFR >= 30 along with Ace/ARB use for approving Voyxact?

**Answer:** **3 strictly commercial plans** require both **eGFR ≥30 mL/min/1.73 m²** and an **ACE inhibitor/ARB-related requirement** for initial Voyxact approval:

1. **UnitedHealthcare Commercial** — eGFR ≥30 plus concomitant maximally tolerated ACE inhibitor or ARB therapy, with an intolerance/contraindication alternative. [02_uhc_commercial, page 1, A. Initial Authorization; page 2, -OR-]
2. **Cigna Commercial** — eGFR ≥30 plus maximum or maximally tolerated ACE inhibitor or ARB therapy for at least 12 weeks. [05_cigna_commercial, page 3, FDA-Approved Indication]
3. **Highmark Wyoming Commercial** — eGFR ≥30 plus at least 3 months of maximally tolerated ACEi/ARB therapy, with specified exceptions. [13_highmark_wy, page 1, PRIOR AUTHORIZATION CLINICAL CRITERIA FOR APPROVAL]

**Unclassified plans with both criteria:** Carelon and Sentara, but they are not counted as Commercial because the retrieved package does not classify them as such. [03_carelon, page 1, Voyxact (sibeprenlimab-szsi); 08_sentara, pages 1–2, Initial Authorization: 9 months / PA Voyxact (CORE)]

### Question 3: What is the typical Medicaid UPCR requirement?

**Answer:** Across the five Medicaid-classified documents with an operative UPCR criterion, the typical initial requirement is **UPCR ≥0.75–0.8 g/g**, usually based on a 24-hour urine collection:

- **0.75 g/g:** Wisconsin Medicaid and Alaska Medicaid [01_wi_medicaid, p. 1, “Clinical Criteria for Voyxact”; 06_ak_medicaid, p. 1, “APPROVAL CRITERIA 1,2,3,4”]
- **0.8 g/g:** Rhode Island Medicaid and Aetna Better Health’s Medicaid policy [07_ri_medicaid, p. 1, “Primary immunoglobulin A nephropathy (IgAN)”; 12_aetna_commercial, p. 2, “Primary Immunoglobulin A Nephropathy (IgAN) 1-4”]
- **0.5 g/g:** Ventura County Health Care Plan [14_vchcp_ca_medicaid, p. 2, “FDA-Approved Indication”]

Thus, the observed Medicaid UPCR range is **≥0.5 to ≥0.8 g/g**, with the main concentration at **≥0.75–0.8 g/g**. The policies generally allow proteinuria measured in g/day as an alternative to the UPCR threshold. Iowa Medicaid was selected, but its retrieved document contains **no operative UPCR requirement**; its UPCR text is contextual trial information rather than a PA criterion [10_ia_medicaid, p. 3, “Clinical Efficacy Summary”].

### Question 4: Which are the most stringent plans in terms of PA criteria for Voyxact?

**Answer:** ## Strict Commercial result

The **most stringent Commercial plan is BCBS Michigan/Blue Care Network**, based on the breadth of its operative step-therapy requirements. It requires:

- Adult IgAN indication
- Prior trial and failure, contraindication, or intolerance to an SGLT2 inhibitor
- Prior trial and failure, contraindication, or intolerance to Tarpeyo
- Prior trial and failure, contraindication, or intolerance to a preferred endothelin receptor antagonist
- ACE inhibitor/ARB combination therapy unless contraindicated  
([09_bcbs_mi, page 1, section “Voyxact”])

The next strongest Commercial candidate is **UnitedHealthcare Commercial**, which requires biopsy-confirmed IgAN, disease-progression risk, eGFR ≥30, ACE inhibitor/ARB therapy or an exception, a nephrologist, and failure/intolerance/contraindication to a glucocorticoid—specifically a 30-day trial when failure is asserted. ([02_uhc_commercial, page 1, section “A. Initial Authorization”]; [02_uhc_commercial, page 2, section “-AND-”])

**Cigna Commercial** is also relatively stringent because it adds several cumulative requirements: age ≥18, biopsy confirmation, high-risk disease defined through proteinuria plus optimized ACE inhibitor/ARB therapy for at least 12 weeks, at least 3 months of optimized supportive care, eGFR ≥30, and nephrologist involvement. ([05_cigna_commercial, page 3, section “FDA-Approved Indication”])

### Commercial stringency tiers

1. **BCBS Michigan/BCN — strongest Commercial candidate**
   - Multiple required prior-therapy pathways, including SGLT2 inhibitor, Tarpeyo, and a preferred endothelin receptor antagonist.  
   ([09_bcbs_mi, page 1, section “Voyxact”])

2. **UnitedHealthcare Commercial and Cigna Commercial — high stringency**
   - UHC: glucocorticoid step requirement plus ACE/ARB-related criteria and specialist involvement.  
   ([02_uhc_commercial, page 2, section “-AND-”])
   - Cigna: optimized supportive care, 12-week ACE/ARB therapy, disease-risk criteria, eGFR, age, and nephrologist requirement.  
   ([05_cigna_commercial, page 3, section “FDA-Approved Indication”])

3. **Highmark Wyoming — moderate-to-high stringency**
   - Requires biopsy-confirmed IgAN, eGFR ≥30, at least 3 months of maximally tolerated ACEi/ARB therapy with inadequate response or an exception, and—when applicable—failure/intolerance/contraindication to a preferred agent. ([13_highmark_wy, page 1, section “Initial Evaluation > Target Agent(s) will be approved when”])

4. **CVS Caremark and Western Health Advantage — lower relative stringency**
   - CVS requires biopsy confirmation, proteinuria/UPCR, and 3 months of RAS-inhibitor therapy unless contraindicated. ([04_cvs_commercial, page 2, section “Primary Immunoglobulin A Nephropathy (IgAN) 1-4”])
   - WHA requires IgAN, risk of progression, nephrologist involvement, and a 90-day ACE/ARB trial or exception. ([11_wha_commercial, page 1, section “Voyxact”])

## Relevant Medicaid and Unclassified plans

Among the plans not in the strict Commercial set, **Rhode Island Medicaid and Sentara are the strongest overall stringency candidates**.

- **Rhode Island Medicaid** requires all of the following at initial authorization: nephrologist, age ≥18, biopsy confirmation, proteinuria/UPCR documentation, eGFR ≥30, at least 3 months of RAS-inhibitor therapy or exception, at least 3 months of SGLT2-inhibitor therapy or exception, a documented inadequate response/intolerance/contraindication to a 30-day oral glucocorticoid trial, several combination-therapy exclusions, and no dialysis or kidney transplant. ([07_ri_medicaid, page 1, section “Primary immunoglobulin A nephropathy (IgAN)”])

- **Sentara** requires biopsy-proven IgAN, age ≥18, a nephrologist, 90 days of RAAS therapy, current laboratory documentation, eGFR and proteinuria thresholds, and unsuccessful 3-month trials of Vanrafia or Filspari **and** Tarpeyo, with supporting chart notes or laboratory results. ([08_sentara, page 1, section “Initial Authorization: 9 months”]; [08_sentara, page 2, section “PA Voyxact (CORE)”])

- **Alaska Medicaid** is also relatively stringent, requiring age, nephrologist involvement, biopsy-confirmed IgAN, eGFR ≥30, proteinuria/UPCR, 90 days of ACEI/ARB therapy or documented contraindication/adverse reaction to both, and exclusions for systemic immunosuppressant use and active infection. ([06_ak_medicaid, page 1, section “APPROVAL CRITERIA 1,2,3,4”]; [06_ak_medicaid, page 1, section “DENIAL CRITERIA 1”])

**Carelon** is more stringent than the simpler Commercial policies in some respects—it requires biopsy documentation, proteinuria, eGFR ≥30, ACE/ARB trial or exception, and combination-use documentation—but it does not show the broader multi-drug step requirements found in Rhode Island, Sentara, or BCBS Michigan. ([03_carelon, page 1, section “Voyxact (sibeprenlimab-szsi)”])

### Bottom line

- **Most stringent Commercial plan:** **BCBS Michigan/BCN**
- **Other highly stringent Commercial plans:** **UnitedHealthcare Commercial** and **Cigna Commercial**
- **Most stringent non-Commercial/Unclassified plans:** **Rhode Island Medicaid** and **Sentara**
- **Additional relatively stringent plan:** **Alaska Medicaid**

This is a **qualitative ranking**, not an exact mathematical ordering. The retrieval package states that criterion-row and field counts are only proxies because the complete Boolean logic is not stored; nested AND/OR structures must be interpreted from the source text. Iowa Medicaid has contextual material but **no retrieved operative PA criteria**, so it cannot be ranked.

In [ ]:
questions2 = [
      "What policies have Michigan defined among them?",
      "Which policies cover premenopausal females?",
      "What coverage rules apply to Vyleesi?",
      "Which policies require patients to be at least 18 years old?",
      "Which policies mention automated approval based on claims history?",
      "Which policies require nephrologist involvement?",
      "Which policies require a 30-day glucocorticoid trial?",
      "Which documents are silent on eGFR?",
      "Which policies state both eGFR ≥30 and proteinuria ≥0.5 g/day?",
      "Which policy documents establish Medicaid LOB?",
      "Which policies were revised in 2026?",
      "Which policies apply in Wisconsin?",
      "Which policies do not require prior authorization?",
      "Which policies cover pregnancy?",
      "Which Michigan policies affect the most covered lives?",
      "Which nephrologist-requiring policies affect the most lives?",
      "Rank policies with automated approval pathways by covered lives.",
      "Which Commercial policies require ACE/ARB and how many lives map to them?",
      "For the Michigan policy, give plan count and coverage criteria.",
      "Which eGFR ≥30 and UPCR-qualified policies affect the most lives?",
      "Which policies cover Vyleesi and what are their payers’ current lives?",
      "Which Medicaid policies map to the largest covered-lives footprint?",
      "Does the most restrictive policy also cover the most lives?",
      "Which policy restrictions affect the largest number of plans in Michigan?",
  ]


question_answers2 = ask_policy_questions(questions2)

for index,item in enumerate(question_answers2, start=1):
    display(Markdown(f"### Question {index}: {item['question']}"))
    display(Markdown(f"**Answer:** {item['answer']}"))

### Question 1: What policies have Michigan defined among them?

**Answer:** Among the retrieved **Commercial** policies, Michigan is represented by **Blue Cross Blue Shield of Michigan / Blue Care Network (BCBSM/BCN)**.

### BCBSM/BCN policy: Voyxact prior authorization and step therapy
The policy requires:

- Use for **adults with primary IgA nephropathy (IgAN) at risk for disease progression**, to reduce proteinuria.
- Age **≥18 years**.
- Trial and failure of maximally tolerated **ACE inhibitor or ARB therapy**, unless contraindicated.
- Trial and failure, contraindication, or intolerance to:
  - An **SGLT2 inhibitor**;
  - **Tarpeyo**; and
  - A preferred **endothelin receptor antagonist**.
- Voyxact must be used in combination with an **ACE inhibitor or ARB**, unless contraindicated.
- **Initial approval: 1 year.** [09_bcbs_mi, page 1, section “Voyxact”]

### Renewal
Renewal requires that the current criteria remain met and that the medication is providing **clinical benefit**. The retrieved policy does not provide a separate operative renewal duration. [09_bcbs_mi, page 1, section “Voyxact”]

No other retrieved policy is explicitly identified as a Michigan policy.

### Question 2: Which policies cover premenopausal females?

**Answer:** **None of the retrieved policies can be confirmed as covering premenopausal females.**

The verified package does not contain an operative criterion—or an evidence quote—mentioning **“premenopausal females,” female status, or menopausal status** for any selected policy. The retrieved criteria instead address IgAN diagnosis, age, kidney function, proteinuria, prior therapy, specialist involvement, and related requirements. For example, Cigna requires age ≥18 but does not specify menopausal status [05_cigna_commercial, p. 3, FDA-Approved Indication]; Alaska Medicaid refers only to the FDA-labeled age [06_ak_medicaid, p. 1, APPROVAL CRITERIA 1,2,3,4].

Thus, the package supports **no policy-specific affirmative answer** for premenopausal females. This is an evidence gap, not a determination that such patients are excluded.

### Question 3: What coverage rules apply to Vyleesi?

...

**Answer:** The verified retrieval package does **not contain covered-lives figures or rankings**, so it cannot determine which Medicaid policies map to the largest covered-lives footprint.

The package identifies these Medicaid-classified policies:

- Wisconsin Medicaid — `01_wi_medicaid`
- Alaska Medicaid — `06_ak_medicaid`
- Rhode Island Medicaid — `07_ri_medicaid`
- Iowa Medicaid — `10_ia_medicaid`
- Aetna Better Health — `12_aetna_commercial` (retrieval classification: Medicaid)
- Ventura County Health Care Plan — `14_vchcp_ca_medicaid`

However, none of the retrieved criteria or evidence fields reports enrollment, membership, or covered lives. Therefore, **no largest-footprint policy can be established from the supplied evidence**.

### Question 23: Does the most restrictive policy also cover the most lives?

**Answer:** **No determination can be made from this retrieval package.**

- The package contains prior-authorization criteria, but **no enrollment, membership, or covered-lives data** for any plan. Therefore, it cannot identify which policy covers the most lives.
- It also does not support a definitive ranking of “most restrictive.” The graph explicitly warns that criterion-row counts are only proxies and that the complete Boolean logic is not stored. For example, Rhode Island Medicaid includes ACE/ARB therapy, SGLT2-inhibitor therapy, a glucocorticoid trial or exception, and exclusions on dialysis, transplant, and combination therapy [07_ri_medicaid, page 1, section “Primary immunoglobulin A nephropathy (IgAN)”]. Sentara includes unsuccessful trials of Vanrafia or Filspari **and** Tarpeyo, along with multiple documentation and exclusion requirements [08_sentara, page 2, section “PA Voyxact (CORE)”]. These examples indicate substantial restrictions but do not establish an exact policy ranking.
- The two plans identified as **Unclassified**—Carelon and Sentara—must not be treated as Commercial or Medicaid without additional evidence.

**Conclusion:** The package cannot answer whether the most restrictive policy also covers the most lives. That requires separate plan-enrollment or covered-lives data and a defined, Boolean-aware method for ranking policy restrictiveness.

### Question 24: Which policy restrictions affect the largest number of plans in Michigan?

**Answer:** The package contains **only one operative Michigan policy**: **BCBS Michigan/Blue Care Network**, classified as **Commercial**. Therefore, no restriction can be shown to affect more than **one Michigan plan** in this retrieval set, and a true largest-number comparison across Michigan plans is not possible.

Within the BCBS Michigan policy, the restrictions affecting that one plan include:

- **ACE inhibitor/ARB step therapy:** trial and failure of a maximally tolerated ACE inhibitor or ARB, unless contraindicated.
- **Additional step therapy:** trial and failure, contraindication, or intolerance to:
  - an **SGLT2 inhibitor**;
  - **Tarpeyo**; and
  - a **preferred endothelin receptor antagonist**.
- **Combination-therapy requirement:** Voyxact must be used with an ACE inhibitor or ARB unless contraindicated.
- **Age and indication restriction:** use for adults at least 18 years old with primary IgAN at risk for disease progression.
- **Renewal restriction:** current criteria must continue to be met and the medication must provide clinical benefit.  
  [09_bcbs_mi, page 1, section “Voyxact”]

The package does not provide a separate Michigan Medicaid policy. Aetna’s document explicitly lists **Michigan as unchecked**, so it should not be counted as a Michigan plan. [12_aetna_commercial, page 1, section “AETNA BETTER HEALTH®”]

In [14]:
def save_question_answers_markdown(
    question_answers,
    output_file="policy_question_answers2.md",
    max_records=100,
    max_answer_characters=50000,
):
    """Validate, sanitize, and atomically save question/answer Markdown."""
    import tempfile
    from pathlib import Path

    if not isinstance(question_answers, list) or not question_answers:
        raise ValueError("question_answers must be a non-empty list")
    if len(question_answers) > max_records:
        raise ValueError(f"At most {max_records} records may be written")

    output_path = Path(output_file).resolve()
    workspace_root = Path.cwd().resolve()
    if output_path.suffix.casefold() != ".md":
        raise ValueError("output_file must use the .md extension")
    if output_path != workspace_root and workspace_root not in output_path.parents:
        raise ValueError("Markdown output must remain inside the current workspace")

    def sanitize_question(value):
        text = re.sub(r"\s+", " ", str(value or "")).strip()
        if not text:
            raise ValueError("Every record requires a non-empty question")
        return text.replace("#", r"\#")

    def sanitize_answer(value):
        text = str(value or "").strip()
        if not text:
            raise ValueError("Every record requires a non-empty answer")
        if len(text) > max_answer_characters:
            raise ValueError("An answer exceeds the configured Markdown size limit")
        text = re.sub(
            r"(?is)<(script|iframe|object|embed)\b.*?>.*?</\1\s*>",
            "",
            text,
        )
        text = re.sub(r"(?s)<[^>]+>", "", text)
        text = re.sub(r"(?i)javascript\s*:", "", text)
        return text

    sections = []
    for index, item in enumerate(question_answers, start=1):
        if not isinstance(item, dict) or set(item) != {"question", "answer"}:
            raise ValueError("Each record must contain only question and answer")
        question = sanitize_question(item["question"])
        answer = sanitize_answer(item["answer"])
        sections.append(
            f"## Question {index}\n\n"
            f"{question}\n\n"
            f"### Answer\n\n"
            f"{answer}\n"
        )

    markdown_output = "\n\n---\n\n".join(sections)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    temporary_name = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            suffix=".tmp",
            delete=False,
        ) as file:
            temporary_name = file.name
            file.write(markdown_output)
        Path(temporary_name).replace(output_path)
    finally:
        if temporary_name and Path(temporary_name).exists():
            Path(temporary_name).unlink()

    return output_path


output_path = save_question_answers_markdown(
    question_answers2,
    output_file="policy_question_answers2.md",
)
print(f"Markdown saved to: {output_path}")

## Question 1

What policies have Michigan defined among them?

### Answer

Among the retrieved **Commercial** policies, Michigan is represented by **Blue Cross Blue Shield of Michigan / Blue Care Network (BCBSM/BCN)**.

### BCBSM/BCN policy: Voyxact prior authorization and step therapy
The policy requires:

- Use for **adults with primary IgA nephropathy (IgAN) at risk for disease progression**, to reduce proteinuria.
- Age **≥18 years**.
- Trial and failure of maximally tolerated **ACE inhibitor or ARB therapy**, unless contraindicated.
- Trial and failure, contraindication, or intolerance to:
  - An **SGLT2 inhibitor**;
  - **Tarpeyo**; and
  - A preferred **endothelin receptor antagonist**.
- Voyxact must be used in combination with an **ACE inhibitor or ARB**, unless contraindicated.
- **Initial approval: 1 year.** [09_bcbs_mi, page 1, section “Voyxact”]

### Renewal
Renewal requires that the current criteria remain met and that the medication is providing **clinical benefit**. The retrieved policy does not provide a separate operative renewal duration. [09_bcbs_mi, page 1, section “Voyxact”]

No other retrieved policy is explicitly identified as a Michigan policy.


## Question 2

Which policies cover premenopausal females?

### Answer

**None of the retrieved policies can be confirmed as covering premenopausal females.**

The verified package does not contain an operative criterion—or an evidence quote—mentioning **“premenopausal females,” female status, or menopausal status** for any selected policy. The retrieved criteria instead address IgAN diagnosis, age, kidney function, proteinuria, prior therapy, specialist involvement, and related requirements. For example, Cigna requires age ≥18 but does not specify menopausal status [05_cigna_commercial, p. 3, FDA-Approved Indication]; Alaska Medicaid refers only to the FDA-labeled age [06_ak_medicaid, p. 1, APPROVAL CRITERIA 1,2,3,4].

Thus, the package supports **no policy-specific affirmative answer** for premenopausal females. This is an evidence gap, not a determination that such patients are excluded.


## Question 3

What coverage rules apply to Vyleesi?

### Answer

## Vyleesi coverage rule identified

The verified package contains one operative policy specifically for **Vyleesi**: **Blue Cross Blue Shield of Michigan / Blue Care Network (Commercial)**. The other retrieved policies concern **Voyxact (sibeprenlimab-szsi)**, a different drug, and should not be applied to Vyleesi.

### Initial authorization

Coverage requires all of the following:

- The member is a **premenopausal female age 18 or older**.
- Diagnosis of **acquired, generalized hypoactive sexual desire disorder (HSDD)** lasting **more than 6 months**.
- Other causes of HSDD—such as **relationship difficulty, substance abuse, or medication side effects**—have been ruled out.  
  [09_bcbs_mi, page 1, section “Document”]

**Initial authorization duration:** 60 days.  
[09_bcbs_mi, page 1, section “Document”]

### Renewal

Renewal requires that:

- The current criteria continue to be met; and
- Vyleesi is providing **clinical benefit**.  
[09_bcbs_mi, page 1, section “Document”]

The retrieved Vyleesi text does not specify a separate renewal duration.


## Question 4

Which policies require patients to be at least 18 years old?

### Answer

Policies with an explicit requirement that the patient be at least 18 years old are:

**Commercial**
- **Cigna** — “Patient is ≥ 18 years of age.” [05_cigna_commercial, page 3, “FDA-Approved Indication”]
- **Blue Cross Blue Shield of Michigan / Blue Care Network** — “Age ≥ 18 years old.” [09_bcbs_mi, page 1, “Voyxact”]

**Medicaid**
- **Rhode Island Medicaid / Neighborhood Health Plan of Rhode Island** — “Member is 18 years of age or older.” [07_ri_medicaid, page 1, “Primary immunoglobulin A nephropathy (IgAN)”]
- **Ventura County Health Care Plan (VCHCP)** — “Patient is ≥ 18 years of age.” [14_vchcp_ca_medicaid, page 2, “FDA-Approved Indication”]

**Unclassified plans**
- **Sentara Health Plans** — “Member is 18 years of age or older.” [08_sentara, page 1, “Initial Authorization: 9 months”]

Alaska Medicaid requires the patient to meet the **FDA-labeled age**, but the retrieved criterion does not explicitly state “at least 18”; therefore, it is not counted as an explicit ≥18 requirement. Highmark Wyoming similarly allows either age within FDA labeling **or** support for use at the patient’s age, so it is not an explicit ≥18 requirement.


## Question 5

Which policies mention automated approval based on claims history?

### Answer

**UnitedHealthcare Commercial** is the only policy in the retrieved package that mentions automated approval based on claims history. It states that UnitedHealthcare may approve **initial authorization and reauthorization** based solely on previous claim or medication history, diagnosis codes, and/or claim logic; automated approval and reapproval vary by program or therapeutic class. **[02_uhc_commercial, page 2, “3. Additional Clinical Rules:”]**


...

## Question 21

Which policies cover Vyleesi and what are their payers’ current lives?

### Answer

## Vyleesi coverage identified

The only retrieved policy that addresses **Vyleesi** is:

- **Blue Cross Blue Shield of Michigan / Blue Care Network (BCBSM/BCN)** — classified as **Commercial**. The policy covers Vyleesi for:
  - Premenopausal females aged 18 or older
  - Acquired, generalized HSDD lasting more than six months
  - Other causes of HSDD ruled out  
  Initial approval is **60 days**; renewal requires current criteria and clinical benefit. **[09_bcbs_mi, page 1, Document]**

## Payer current lives

The verified retrieval package does **not provide membership or “current lives” counts** for BCBSM/BCN—or for any other payer. Therefore, no supported current-lives figure can be reported.

The other retrieved policies concern **Voyxact**, not Vyleesi, and should not be counted as Vyleesi coverage.


## Question 22

Which Medicaid policies map to the largest covered-lives footprint?

### Answer

The verified retrieval package does **not contain covered-lives figures or rankings**, so it cannot determine which Medicaid policies map to the largest covered-lives footprint.

The package identifies these Medicaid-classified policies:

- Wisconsin Medicaid — `01_wi_medicaid`
- Alaska Medicaid — `06_ak_medicaid`
- Rhode Island Medicaid — `07_ri_medicaid`
- Iowa Medicaid — `10_ia_medicaid`
- Aetna Better Health — `12_aetna_commercial` (retrieval classification: Medicaid)
- Ventura County Health Care Plan — `14_vchcp_ca_medicaid`

However, none of the retrieved criteria or evidence fields reports enrollment, membership, or covered lives. Therefore, **no largest-footprint policy can be established from the supplied evidence**.


## Question 23

Does the most restrictive policy also cover the most lives?

### Answer

**No determination can be made from this retrieval package.**

- The package contains prior-authorization criteria, but **no enrollment, membership, or covered-lives data** for any plan. Therefore, it cannot identify which policy covers the most lives.
- It also does not support a definitive ranking of “most restrictive.” The graph explicitly warns that criterion-row counts are only proxies and that the complete Boolean logic is not stored. For example, Rhode Island Medicaid includes ACE/ARB therapy, SGLT2-inhibitor therapy, a glucocorticoid trial or exception, and exclusions on dialysis, transplant, and combination therapy [07_ri_medicaid, page 1, section “Primary immunoglobulin A nephropathy (IgAN)”]. Sentara includes unsuccessful trials of Vanrafia or Filspari **and** Tarpeyo, along with multiple documentation and exclusion requirements [08_sentara, page 2, section “PA Voyxact (CORE)”]. These examples indicate substantial restrictions but do not establish an exact policy ranking.
- The two plans identified as **Unclassified**—Carelon and Sentara—must not be treated as Commercial or Medicaid without additional evidence.

**Conclusion:** The package cannot answer whether the most restrictive policy also covers the most lives. That requires separate plan-enrollment or covered-lives data and a defined, Boolean-aware method for ranking policy restrictiveness.


## Question 24

Which policy restrictions affect the largest number of plans in Michigan?

### Answer

The package contains **only one operative Michigan policy**: **BCBS Michigan/Blue Care Network**, classified as **Commercial**. Therefore, no restriction can be shown to affect more than **one Michigan plan** in this retrieval set, and a true largest-number comparison across Michigan plans is not possible.

Within the BCBS Michigan policy, the restrictions affecting that one plan include:

- **ACE inhibitor/ARB step therapy:** trial and failure of a maximally tolerated ACE inhibitor or ARB, unless contraindicated.
- **Additional step therapy:** trial and failure, contraindication, or intolerance to:
  - an **SGLT2 inhibitor**;
  - **Tarpeyo**; and
  - a **preferred endothelin receptor antagonist**.
- **Combination-therapy requirement:** Voyxact must be used with an ACE inhibitor or ARB unless contraindicated.
- **Age and indication restriction:** use for adults at least 18 years old with primary IgAN at risk for disease progression.
- **Renewal restriction:** current criteria must continue to be met and the medication must provide clinical benefit.  
  [09_bcbs_mi, page 1, section “Voyxact”]

The package does not provide a separate Michigan Medicaid policy. Aetna’s document explicitly lists **Michigan as unchecked**, so it should not be counted as a Michigan plan. [12_aetna_commercial, page 1, section “AETNA BETTER HEALTH®”]


Markdown saved to: C:\Users\RahulKumar\Documents\GDB\policy_question_answers2.md


In [ ]:
result = ask_policy_graph(
    "How many commercial plans need eGFR >= 30 along with Ace/ARB use for approving Voyxact?",
    show_raw_result=True,
)
print(result["answer"])

In [ ]:
result = ask_policy_graph(
    "What is the typical UPCR test range requirement imposed by medciad plans?",
    show_raw_result=True,
)
print(result["answer"])

In [ ]:
result = ask_policy_graph(
    "What is the PA criteria for UHC commercial for Voyxact?",
    show_raw_result=True,
)
print(result["answer"])

In [ ]:
result = ask_policy_graph(
    "Which are the most stringent plans in terms of PA criteria for Voyxact? (most number of “AND” conditions?",
    show_raw_result=True,
)
print(result["answer"])

In [ ]:
result = ask_policy_graph(
    "Show the source evidence for UHC step therapy requirements",
    show_raw_result=True,
)
print(result["answer"])

In [ ]:
result = ask_policy_graph(
    "What are the UPCR requirements for Voyxact?",
    show_raw_result=True,
)
print(result["answer"])